# Meeting Notes Assistant - Powered by memweave

An interactive assistant that answers questions across your meeting notes and learns from every conversation through natural dialogue.

## How it works

| Step | What happens |
|---|---|
| **Startup** | Indexes all `.md` files in `workspace/memory/` - skips unchanged files |
| **Each turn** | Searches memory for relevant context, answers using that context |
| **On exit** | Flushes the session - extracts durable facts to today's memory file |

## Why this matters

- **`search()`** - sub-100ms hybrid BM25 + vector search across all your notes
- **`flush()`** - no need to write `.md` files manually - just describe your meeting
- **`index()`** - files already indexed are skipped - instant startup on return visits
- Memory is plain `.md` files - readable, editable, git-diffable

## Try it

```
Ask about past meetings:   "What action items are still open?"
Ask about budget:          "What was the approved budget?"
Ask about risks:           "What risks have been raised so far?"
Describe today's meeting:  "We just decided to cut the analytics feature..."
Then type 'exit' - the facts are saved and searchable next session.
```

## 1. Installation

Install `memweave`

Set your API key as an environment variable before running - memweave works with any LiteLLM-compatible provider:

```bash
# OpenAI
export OPENAI_API_KEY="sk-..."

# Gemini
export GEMINI_API_KEY="..."

# Anthropic
export ANTHROPIC_API_KEY="sk-ant-..."
```

In [ ]:
%pip install memweave

In [ ]:
# openai-api key example
import os
os.environ["OPENAI_API_KEY"] = ""

## 2. Imports & Configuration

Memweave defaults to `text-embedding-3-small` for embeddings and `gpt-4o-mini` for flush - no extra config needed.
Swap `LLM_MODEL` for any [LiteLLM-supported model](https://docs.litellm.ai/docs/providers)
(e.g. `"gemini/gemini-2.5-flash"`).

In [ ]:
from __future__ import annotations

from datetime import date, timedelta
from pathlib import Path

import litellm

from memweave import FlushConfig, MemoryConfig, MemWeave, QueryConfig

LLM_MODEL   = "gpt-4o-mini"  # swap for any LiteLLM-compatible model
WORKSPACE   = Path.cwd() / "workspace"
MAX_HISTORY = 10  # last 5 turns kept in prompt context
MAX_RESULTS = 3   # search results per query

SYSTEM_PROMPT = (
    "You are a helpful meeting notes assistant for a project manager. "
    "Answer questions using only the meeting notes context provided. "
    "Cite the source file name where relevant. "
    "If the context is insufficient, say so clearly."
)

## 3. Create Sample Dataset

This cell generates 5 realistic meeting notes into `workspace/memory/notes/` with dates relative to **today**.

| File | Represents | Age |
|---|---|---|
| `{today-25}_kickoff.md` | Project kickoff - team roles, initial tasks | 25 days ago |
| `{today-20}_design_review.md` | Design decisions, chart library risk | 20 days ago |
| `{today-15}_budget.md` | $45,000 budget approved, breakdown | 15 days ago |
| `{today-7}_sprint_planning.md` | Sprint 1 tasks, payment API risk | 7 days ago |
| `{today-3}_retro.md` | Sprint 1 retro - blockers, **open action items** | 3 days ago |

Action items with future due dates are in the last two files - so *"What action items are still open?"* always returns meaningful results no matter when you run this notebook.

In [ ]:
today = date.today()

d_kickoff = today - timedelta(days=25)
d_design  = today - timedelta(days=20)
d_budget  = today - timedelta(days=15)
d_sprint  = today - timedelta(days=7)
d_retro   = today - timedelta(days=3)

due_soon   = (today + timedelta(days=2)).strftime("%B %-d")
due_mid    = (today + timedelta(days=5)).strftime("%B %-d")
due_later  = (today + timedelta(days=9)).strftime("%B %-d")
due_sprint = (today + timedelta(days=4)).strftime("%B %-d")

NOTES_DIR = WORKSPACE / "memory" / "notes"
NOTES_DIR.mkdir(parents=True, exist_ok=True)

notes = {
    f"{d_kickoff.isoformat()}_kickoff.md": (
        f"# Project Kickoff - {d_kickoff.strftime('%B %-d, %Y')}\n\n"
        "**Attendees:** Sarah (PM), Alice (Frontend), Mike (Backend), Lisa (Design)\n\n"
        "## Goals\n\n"
        "- Build a customer portal with dashboard, settings, and payment integration\n"
        "- Target launch: end of next month\n"
        "- Stack: React frontend, Python backend, Stripe for payments\n\n"
        "## Roles\n\n"
        "| Person | Role |\n"
        "|--------|------|\n"
        "| Sarah  | Project manager - owns timeline and stakeholder communication |\n"
        "| Alice  | Frontend lead - React, design tokens, charts |\n"
        "| Mike   | Backend lead - API, database, payment integration |\n"
        "| Lisa   | Design lead - wireframes, hi-fi mockups, design system |\n\n"
        "## Action Items\n\n"
        "| Owner | Task | Due |\n"
        "|-------|------|-----|\n"
        f"| Sarah | Finalise project charter | {(d_kickoff + timedelta(days=3)).strftime('%B %-d')} |\n"
        f"| Mike  | Set up backend repo and CI pipeline | {(d_kickoff + timedelta(days=5)).strftime('%B %-d')} |\n"
        f"| Alice | Create frontend scaffolding | {(d_kickoff + timedelta(days=5)).strftime('%B %-d')} |\n"
        f"| Lisa  | Deliver initial wireframes | {(d_kickoff + timedelta(days=7)).strftime('%B %-d')} |\n"
    ),
    f"{d_design.isoformat()}_design_review.md": (
        f"# Design Review - {d_design.strftime('%B %-d, %Y')}\n\n"
        "**Attendees:** Sarah, Alice, Lisa\n\n"
        "## Decisions\n\n"
        "- Adopted a design token system for colours and typography\n"
        "- Dashboard will use Recharts - Alice to evaluate theming support\n"
        "- Settings screen approved; Lisa to deliver hi-fi mockups by end of week\n\n"
        "## Risks\n\n"
        "- Third-party chart library may not support custom theming - Alice will evaluate alternatives\n\n"
        "## Action Items\n\n"
        "| Owner | Task | Due |\n"
        "|-------|------|-----|\n"
        f"| Alice | Implement design tokens in frontend | {(d_design + timedelta(days=2)).strftime('%B %-d')} |\n"
        f"| Alice | Evaluate chart library theming | {(d_design + timedelta(days=4)).strftime('%B %-d')} |\n"
        f"| Lisa  | Deliver hi-fi mockups for settings screen | {(d_design + timedelta(days=3)).strftime('%B %-d')} |\n"
    ),
    f"{d_budget.isoformat()}_budget.md": (
        f"# Budget Approval - {d_budget.strftime('%B %-d, %Y')}\n\n"
        "**Attendees:** Sarah, Head of Engineering, Finance\n\n"
        "## Budget Approved: $45,000 total\n\n"
        "| Category | Amount |\n"
        "|----------|--------|\n"
        "| Development | $20,000 |\n"
        "| Infrastructure | $10,000 |\n"
        "| Design and UX | $8,000 |\n"
        "| Marketing and launch | $7,000 |\n\n"
        "## Notes\n\n"
        "- Infrastructure budget covers cloud hosting for 12 months post-launch\n"
        "- Any scope changes require Head of Engineering approval\n"
        "- Sarah to track actuals vs budget weekly\n"
    ),
    f"{d_sprint.isoformat()}_sprint_planning.md": (
        f"# Sprint 1 Planning - {d_sprint.strftime('%B %-d, %Y')}\n\n"
        "**Attendees:** Sarah, Alice, Mike, Lisa\n\n"
        "## Sprint Goal\n\n"
        "Deliver a working dashboard with charts and the settings screen by end of sprint.\n\n"
        "## Risks\n\n"
        "- Payment provider API docs appear outdated - Mike flagged inconsistencies\n"
        "- Lisa has reduced availability due to a conference\n\n"
        "## Action Items\n\n"
        "| Owner | Task | Due |\n"
        "|-------|------|-----|\n"
        f"| Mike  | Confirm payment API sandbox credentials with Stripe | {due_sprint} |\n"
        f"| Sarah | Adjust sprint scope if Lisa's availability drops | {due_sprint} |\n"
    ),
    f"{d_retro.isoformat()}_retro.md": (
        f"# Sprint 1 Retrospective - {d_retro.strftime('%B %-d, %Y')}\n\n"
        "**Attendees:** Sarah, Alice, Mike, Lisa\n\n"
        "## What Went Well\n\n"
        "- Alice delivered all frontend tasks on time; charts look great\n"
        "- Team communication was smooth\n"
        "- Design tokens made frontend theming faster than expected\n\n"
        "## What Didn't Go Well\n\n"
        "- Payment provider API docs were outdated - Mike spent 3 days reverse-engineering\n"
        "- Lisa was sick for 2 days; UI tasks slipped slightly\n\n"
        "## Blockers\n\n"
        "- Outdated payment provider API docs (resolved, but costly in time)\n"
        "- Lisa's illness caused a 2-day slip on checkout screen mockups\n\n"
        "## Action Items\n\n"
        "| Owner | Task | Due |\n"
        "|-------|------|-----|\n"
        f"| Mike  | Escalate outdated payment API docs to vendor | {due_soon} |\n"
        f"| Sarah | Update project timeline to account for Lisa's delay | {due_mid} |\n"
        f"| Alice | Begin API redesign spike - findings due at next sync | {due_later} |\n"
    ),
}

for filename, content in notes.items():
    (NOTES_DIR / filename).write_text(content, encoding="utf-8")

print(f"Created {len(notes)} meeting notes in {NOTES_DIR}")
for filename in sorted(notes):
    print(f"  - {filename}")

## 4. Core Agent Function

`ask_agent` is the heart of the assistant - a single RAG loop:

1. **Retrieve** - `mem.search()` finds the most relevant chunks from all indexed notes
2. **Augment** - the retrieved snippets are injected into the system prompt as context
3. **Generate** - the LLM answers using that context plus the conversation history

No LLM is called for the retrieval step - `mem.search()` is pure BM25 + vector lookup.

In [ ]:
async def ask_agent(
    question: str,
    history: list[dict],
    mem: MemWeave,
) -> str:
    """Search memory for relevant context, then ask the LLM to answer."""
    results = await mem.search(question, max_results=MAX_RESULTS, min_score=0.1)

    context = (
        "\n\n---\n\n".join(
            f"**Source:** `{Path(r.path).name}` (relevance: {r.score:.2f})\n\n{r.snippet}"
            for r in results
        )
        or "No relevant content found in memory."
    )

    messages = [
        {
            "role": "system",
            "content": f"{SYSTEM_PROMPT}\n\n### Meeting Notes Context\n\n{context}",
        },
        *history[-MAX_HISTORY:],
        {"role": "user", "content": question},
    ]

    response = await litellm.acompletion(
        model=LLM_MODEL,
        messages=messages,
        max_tokens=512,
    )
    return response.choices[0].message.content.strip()

## 5. Run the Interactive Agent

Running the cell below starts the agent loop:

- On startup it indexes the meeting notes generated above - subsequent runs skip unchanged files
- Type any question or describe a meeting that just happened
- Type `exit` or `quit` to end the session - memweave will extract any new facts and save them to `workspace/memory/YYYY-MM-DD.md`, making them searchable next time

> **Note:** `input()` works in Jupyter - an input box will appear below the cell after each agent response.

In [ ]:
async def main() -> None:
    today_str = date.today().isoformat()
    config = MemoryConfig(
        workspace_dir=WORKSPACE,
        query=QueryConfig(snippet_max_chars=2000),
        flush=FlushConfig(
            model=LLM_MODEL,
            system_prompt=(
                f"You are a meeting notes assistant. Today's date is {today_str}.\n"
                "Review the conversation and extract ONLY facts that the USER described "
                "as new information (e.g. describing a meeting that just happened).\n"
                "Do NOT extract facts that appeared in the assistant's answers - "
                "those already exist in memory.\n"
                f"If there are new facts, write them as a concise bullet list under "
                f"the heading '# Meeting Notes - {today_str}'.\n"
                "Include: decisions made, action items (owner + deadline), and risks raised.\n"
                "If the conversation was only Q&A with no new information from the user, "
                "reply with @@SILENT_REPLY@@."
            ),
        ),
    )

    async with MemWeave(config) as mem:
        result = await mem.index()
        total = result.files_indexed + result.files_skipped
        if result.files_indexed > 0:
            print(f"Loaded {result.files_indexed} meeting notes. Ask me anything.\n")
        else:
            print(f"Memory ready - {total} notes from previous sessions.\n")

        print("Type your question or describe what happened in a meeting.")
        print("Type 'exit' to save this session to memory and quit.\n")

        history: list[dict] = []

        while True:
            try:
                user_input = input("You: ").strip()
            except (EOFError, KeyboardInterrupt):
                print()
                break

            if not user_input:
                continue
            if user_input.lower() in {"exit", "quit"}:
                break

            answer = await ask_agent(user_input, history, mem)
            print(f"\nAgent: {answer}\n")

            history.append({"role": "user", "content": user_input})
            history.append({"role": "assistant", "content": answer})

        if history:
            print("\nSaving session to memory...")
            extracted = await mem.flush(history)
            if extracted:
                print("\nSession saved. Facts extracted:\n")
                for line in extracted.strip().splitlines():
                    print(f"  {line}")
                dated_file = WORKSPACE / "memory" / f"{today_str}.md"
                print(f"\nWritten to: {dated_file}")
                print("Searchable in the next session - no extra indexing needed.\n")
            else:
                print("Nothing new to save.\n")


await main()